# 🎁 GiftGenius — Kaggle backend на Qwen3.5-4B (4-bit)

Этот ноутбук запускает **весь LLM-инференс без внешнего API** прямо в Kaggle на **одной GPU T4**.  
Локально у вас остается только Streamlit-интерфейс.

Схема:
- **локально**: `app.py` / `app_qwen_local.py`
- **в Kaggle**: `FastAPI + RAG + Qwen3.5-4B 4-bit`
- связь между ними: `ngrok`


In [1]:
!pip install -q -U \
  git+https://github.com/huggingface/transformers \
  accelerate \
  bitsandbytes \
  pandas \
  fastapi \
  uvicorn \
  pyngrok \
  psutil \
  cloudscraper \
  beautifulsoup4 \
  lxml

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 10.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 96.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import torch

# ====== ЗАПОЛНИТЕ ПОД СЕБЯ ======
os.environ["NGROK_AUTH_TOKEN"] = ""
os.environ["DATA_PATH"] = "/kaggle/input/datasets/aleksanderviktorov/gift-genius-dataset/gifts_dataset_ru_plus.csv"

os.environ["MODEL_NAME"] = "Qwen/Qwen3.5-4B"
os.environ["EMBED_DEVICE"] = "cpu"
os.environ["ENABLE_THINKING"] = "false"
os.environ["GEN_TEMPERATURE"] = "0.7"
os.environ["GEN_TOP_P"] = "0.8"
os.environ["GEN_TOP_K"] = "20"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM, GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("❌ GPU не включен. Включите Settings -> Accelerator -> GPU (T4).")

print("DATA_PATH:", os.environ["DATA_PATH"])
print("DATA_PATH exists:", os.path.exists(os.environ["DATA_PATH"]))

if not os.path.exists(os.environ["DATA_PATH"]):
    print("\nФайлы в /kaggle/input:")
    os.system("find /kaggle/input -maxdepth 3 -type f | sed -n '1,80p'")

CUDA available: True
GPU: Tesla T4
VRAM, GB: 14.56
DATA_PATH: /kaggle/input/datasets/aleksanderviktorov/gift-genius-dataset/gifts_dataset_ru_plus.csv
DATA_PATH exists: True


In [3]:
%%writefile rag_pipeline_qwen_kaggle.py
import math
import os
import re
import urllib.parse
from collections import Counter
from typing import List, Tuple

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# =========================
# Конфиг
# =========================
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3.5-4B")
DATA_PATH = os.getenv(
    "DATA_PATH",
    "/kaggle/input/datasets/ursofiia/gift-genius-dataset/gifts_dataset_ru_plus (1).csv",
)
FINAL_TOP_K = int(os.getenv("FINAL_TOP_K", "15"))
ENABLE_THINKING = os.getenv("ENABLE_THINKING", "false").lower() == "true"
LLM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GEN_CONFIG = {
    "temperature": float(os.getenv("GEN_TEMPERATURE", "0.7")),
    "top_p": float(os.getenv("GEN_TOP_P", "0.8")),
    "top_k": int(os.getenv("GEN_TOP_K", "20")),
    "repetition_penalty": float(os.getenv("GEN_REPETITION_PENALTY", "1.05")),
}

print("=" * 60)
print("🎁 GiftGenius | Kaggle backend without external LLM API")
print("=" * 60)
print(f"LLM model: {MODEL_NAME}")
print(f"LLM device: {LLM_DEVICE}")
print(f"Thinking mode: {ENABLE_THINKING}")
print("Retrieval: pure-python BM25 (no sentence-transformers / no scipy)")
print("=" * 60)


def normalize_product_name(name: str) -> str:
    name = re.sub(r"[‑–—]", "-", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def create_market_links(product_name: str) -> dict:
    clean_name = normalize_product_name(product_name.split(",")[0].strip().strip('"'))
    encoded_wb = urllib.parse.quote(clean_name)
    encoded_ya = urllib.parse.quote(clean_name).replace("%20", "+")
    encoded_ozon = urllib.parse.quote(clean_name)
    return {
        "wildberries": f"https://www.wildberries.ru/catalog/0/search.aspx?search={encoded_wb}",
        "yamarket": f"https://market.yandex.ru/search?text={encoded_ya}",
        "ozon": f"https://www.ozon.ru/search/?text={encoded_ozon}",
    }


# =========================
# Data + retrieval
# =========================
print("\n📊 Загрузка датасета...")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"DATA_PATH не найден: {DATA_PATH}")

df = pd.read_csv(DATA_PATH).fillna("")
print(f"✅ Подарков в базе: {len(df)}")

df["_doc"] = df.astype(str).agg(" | ".join, axis=1).str.strip()
corpus = df["_doc"].tolist()


def tokenize(text: str) -> List[str]:
    text = str(text).lower().replace("ё", "е")
    return re.findall(r"[a-zа-я0-9]+", text)


print("\n🔍 Построение BM25 индекса...")
corpus_tokens = [tokenize(doc) for doc in corpus]
doc_lens = [len(tokens) for tokens in corpus_tokens]
avgdl = sum(doc_lens) / max(len(doc_lens), 1)
doc_freq = Counter()
doc_term_freqs = []
for tokens in corpus_tokens:
    tf = Counter(tokens)
    doc_term_freqs.append(tf)
    for tok in tf.keys():
        doc_freq[tok] += 1

N = len(corpus_tokens)
IDF = {}
for tok, dfreq in doc_freq.items():
    IDF[tok] = math.log(1 + (N - dfreq + 0.5) / (dfreq + 0.5))

K1 = 1.5
B = 0.75
print(f"✅ BM25 готов | docs={N} | avgdl={avgdl:.1f}")


_OCCASION_HINTS = {
    "день рождения": ["день", "рождения", "др", "юбилей"],
    "новый год": ["новый", "год", "новогодний"],
    "свадьба": ["свадьба", "молодожены"],
    "8 марта": ["8", "марта"],
    "23 февраля": ["23", "февраля"],
}


def bm25_score(query_tokens: List[str], doc_idx: int) -> float:
    tf = doc_term_freqs[doc_idx]
    dl = doc_lens[doc_idx]
    score = 0.0
    for tok in query_tokens:
        freq = tf.get(tok, 0)
        if freq == 0:
            continue
        idf = IDF.get(tok, 0.0)
        denom = freq + K1 * (1 - B + B * dl / max(avgdl, 1e-9))
        score += idf * (freq * (K1 + 1)) / max(denom, 1e-9)
    return score



def retrieve_for_query(query_text: str, top_k: int = FINAL_TOP_K) -> List[int]:
    query_tokens = tokenize(query_text)
    if not query_tokens:
        return list(range(min(top_k, len(df))))

    scores = []
    raw_query = query_text.lower().replace("ё", "е")
    for i, doc_text in enumerate(corpus):
        score = bm25_score(query_tokens, i)

        # Лёгкие бусты для точных фраз и названий категорий.
        doc_lower = str(doc_text).lower().replace("ё", "е")
        for tok in set(query_tokens):
            if tok in doc_lower:
                score += 0.12
        if raw_query in doc_lower:
            score += 1.0

        scores.append((score, i))

    scores.sort(key=lambda x: x[0], reverse=True)
    return [i for _, i in scores[: max(top_k * 2, 20)]]



def get_products_context(idx_list: List[int], top_k: int = FINAL_TOP_K) -> Tuple[List[str], pd.DataFrame]:
    context_df = df.iloc[idx_list[:top_k]].copy()
    products = []
    for _, row in context_df.iterrows():
        name = str(row.iloc[0]).split(",")[0].strip().strip('"')
        products.append(name)
    return products, context_df


# =========================
# Parsing user request
# =========================
def extract_budget(query: str) -> str | None:
    patterns = [
        r"(?:до|не более|максимум)\s+(\d[\d\s]*)\s*(?:руб|р\b|₽|тыс|k|к)",
        r"(\d[\d\s]*)\s*(?:–|-)\s*(\d[\d\s]*)\s*(?:руб|р\b|₽)",
        r"бюджет[: ]+(\d[\d\s]*)\s*(?:руб|р\b|₽|тыс|k|к)?",
    ]
    for p in patterns:
        m = re.search(p, query, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return None


OCCASION_MAP = {
    "день рождения": "День рождения",
    "новый год": "Новый год",
    "свадьба": "Свадьба",
    "8 марта": "8 марта",
    "23 февраля": "23 февраля",
    "юбилей": "Юбилей",
    "годовщина": "Годовщина",
    "рождение ребенка": "Рождение ребенка",
}



def detect_occasion(query: str) -> str | None:
    q = query.lower()
    for kw, label in OCCASION_MAP.items():
        if kw in q:
            return label
    return None


_RECIPIENT_HINTS = [
    "маме", "папе", "другу", "подруге", "коллеге", "брату", "сестре",
    "мужу", "жене", "дедушке", "бабушке", "ребёнку", "ребенку", "девушке", "парню",
]



def is_query_too_vague(query: str) -> bool:
    words = query.lower().split()
    has_who = any(h in words for h in _RECIPIENT_HINTS)
    return len(words) < 4 and not has_who


# =========================
# LLM loading (4-bit Qwen on one T4)
# =========================
print("\n🤖 Загрузка Qwen3.5-4B в 4-bit...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
print("✅ Qwen загружен")



def _apply_chat_template(messages: list[dict]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


@torch.inference_mode()
def call_local_llm(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 512,
    temperature: float | None = None,
    top_p: float | None = None,
    top_k: int | None = None,
) -> str:
    temperature = GEN_CONFIG["temperature"] if temperature is None else temperature
    top_p = GEN_CONFIG["top_p"] if top_p is None else top_p
    top_k = GEN_CONFIG["top_k"] if top_k is None else top_k

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt = _apply_chat_template(messages)
    inputs = tokenizer(prompt, return_tensors="pt")

    target_device = "cuda:0" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(target_device) for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=GEN_CONFIG["repetition_penalty"],
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    new_tokens = generated_ids[:, inputs["input_ids"].shape[1] :]
    text = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text


# =========================
# Prompts
# =========================
PERSONA_SYSTEM = """Ты — психолог и gift-strategist.
Составь краткий, но полезный портрет получателя подарка.

Нужно описать:
1. возраст/роль/тип получателя, если можно понять из запроса;
2. образ жизни и интересы;
3. что может его порадовать;
4. какие ограничения есть в выборе подарка.

Пиши по-русски, 4-6 предложений, без воды."""



def persona_prompt(query: str, occasion: str | None = None, budget: str | None = None) -> str:
    ctx = ""
    if occasion:
        ctx += f"Повод: {occasion}\n"
    if budget:
        ctx += f"Бюджет: {budget}\n"
    return f'Запрос: "{query}"\n{ctx}\nСделай содержательный портрет человека для подбора подарка.'


ANTI_SYSTEM = """Ты — аналитик по подбору подарков.
На основе портрета выдели категории подарков, которые лучше НЕ дарить.

Верни 3-5 коротких пунктов на русском языке.
Каждый пункт — отдельной строкой и начинается с дефиса."""



def anti_prompt(query: str, persona: str) -> str:
    return f"""ПОРТРЕТ:
{persona}

ИСХОДНЫЙ ЗАПРОС:
{query}

Выдели 3-5 нежелательных категорий подарков."""


CREATIVE_SYSTEM = """Ты — сильный эксперт по подбору подарков.
Сгенерируй 5 качественных идей подарков по-русски.

Обязательные правила:
- идеи должны быть реалистичными и доступными для покупки;
- нельзя предлагать банальности вроде носков, кружек и случайных сувениров;
- названия должны быть конкретными и понятными для поиска на маркетплейсах;
- учитывай характер, интересы, повод и бюджет, если они известны;
- не повторяй одну и ту же категорию подарка разными словами.

Для каждой идеи строго используй формат:
ПОДАРОК: ...
ПОЧЕМУ: ...
МОМЕНТ: ...
ЧЕМ НЕ БАНАЛЬНО: ...
"""



def creative_prompt(
    query: str,
    persona: str,
    anti_list: List[str],
    products: List[str],
    budget: str | None = None,
    occasion: str | None = None,
) -> str:
    products_list = "\n".join([f"- {p}" for p in products[:10]])
    anti_list_text = "\n".join([f"- {item.lstrip('- ').strip()}" for item in anti_list[:5]])

    ctx = ""
    if occasion:
        ctx += f"Повод: {occasion}\n"
    if budget:
        ctx += f"Бюджет: {budget}\n"

    return f"""ПОРТРЕТ ПОЛУЧАТЕЛЯ:
{persona}

НЕЖЕЛАТЕЛЬНЫЕ ПОДАРКИ:
{anti_list_text}

{ctx}РЕФЕРЕНСЫ ИЗ БАЗЫ ТОВАРОВ (не копируй их дословно, используй как ориентир категорий):
{products_list}

ЗАДАЧА:
Придумай 5 разных персонализированных идей подарков для этого запроса: {query}

Начинай сразу с первой идеи и соблюдай формат без дополнительных вступлений."""


# =========================
# Parsing output
# =========================
def parse_ideas(creative_response: str) -> List[str]:
    ideas = []
    current_idea = []

    for raw_line in creative_response.split("\n"):
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith("ПОДАРОК:"):
            if current_idea:
                ideas.append("\n".join(current_idea))
                current_idea = []
            current_idea.append(line)
        elif line.startswith(("ПОЧЕМУ:", "МОМЕНТ:", "ЧЕМ НЕ БАНАЛЬНО:")):
            current_idea.append(line)
        elif current_idea:
            current_idea.append(line)

    if current_idea:
        ideas.append("\n".join(current_idea))

    return ideas[:5]



def extract_idea_title(idea_text: str) -> str:
    for line in idea_text.split("\n"):
        if line.startswith("ПОДАРОК:"):
            return line.replace("ПОДАРОК:", "").strip().strip("*")
    return ""


# =========================
# Main pipeline
# =========================
def gift_agent(query: str, verbose: bool = True):
    if verbose:
        print(f"\n🔍 Обработка запроса: {query}")

    budget = extract_budget(query)
    occasion = detect_occasion(query)

    if is_query_too_vague(query):
        return {
            "status": "needs_clarification",
            "questions": "Уточните, пожалуйста, кому подарок, возраст/роль человека и его интересы.",
        }

    if verbose:
        print("[1/4] Поиск релевантных товаров в базе...")
    idx = retrieve_for_query(query)
    products, _ = get_products_context(idx)

    if verbose:
        print(f"   Найдено референсов: {len(products)}")
        print("[2/4] Создание портрета...")
    persona = call_local_llm(
        PERSONA_SYSTEM,
        persona_prompt(query, occasion, budget),
        max_new_tokens=220,
    )
    if not persona:
        persona = "Получатель любит персональные и осмысленные подарки, подобранные под интересы и образ жизни."

    if verbose:
        print("[3/4] Выделение анти-предпочтений...")
    anti_response = call_local_llm(
        ANTI_SYSTEM,
        anti_prompt(query, persona),
        max_new_tokens=180,
    )
    anti_list = [line.strip() for line in anti_response.split("\n") if line.strip()]
    if not anti_list:
        anti_list = ["- Банальные сувениры", "- Слишком обезличенные подарки"]

    if verbose:
        print("[4/4] Генерация идей подарков...")
    creative_response = call_local_llm(
        CREATIVE_SYSTEM,
        creative_prompt(query, persona, anti_list, products, budget, occasion),
        max_new_tokens=700,
    )

    ideas = parse_ideas(creative_response)
    if not ideas:
        ideas = [
            f"ПОДАРОК: {p}\nПОЧЕМУ: Подходит по интересам и контексту запроса.\nМОМЕНТ: Такой подарок будет уместен и полезен в повседневной жизни.\nЧЕМ НЕ БАНАЛЬНО: Он связан с реальными увлечениями человека, а не выбран случайно."
            for p in products[:5]
        ]

    if verbose:
        print(f"✅ Сгенерировано идей: {len(ideas)}")

    return {
        "status": "success",
        "ideas": ideas,
        "budget": budget,
        "occasion": occasion,
        "persona": persona,
        "anti_list": anti_list,
        "references": products[:10],
    }



def generate_gifts(description: str) -> List[str]:
    try:
        result = gift_agent(description, verbose=False)
        if result["status"] == "needs_clarification":
            return [f"Уточните: {result['questions']}"]
        return result.get("ideas", ["Идеи не найдены"])
    except Exception as e:
        print(f"❌ Ошибка generate_gifts: {e}")
        return [f"Ошибка: {e}"]


print("\n✅ Backend готов: BM25 retrieval на CPU, Qwen 4-bit на GPU")


Writing rag_pipeline_qwen_kaggle.py


In [4]:
%%writefile api_qwen_kaggle.py
from fastapi import FastAPI
from pydantic import BaseModel
import logging
import time

from rag_pipeline_qwen_kaggle import generate_gifts, gift_agent

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="GiftGenius API | Qwen3.5-4B 4bit on Kaggle")


class Request(BaseModel):
    description: str


class Response(BaseModel):
    ideas: list[str]
    processing_time: float | None = None
    status: str = "success"


@app.get("/")
def root():
    return {
        "message": "🎁 GiftGenius v6 | Qwen3.5-4B 4bit on Kaggle",
        "status": "healthy",
    }


@app.get("/health")
def health():
    return {"status": "healthy", "timestamp": time.time()}


@app.post("/generate", response_model=Response)
def generate(request: Request):
    start = time.time()
    logger.info("📨 Запрос: %s", request.description)
    try:
        ideas = generate_gifts(request.description)
        return {
            "ideas": ideas,
            "processing_time": time.time() - start,
            "status": "success",
        }
    except Exception as e:
        logger.exception("❌ Ошибка generate")
        return {
            "ideas": [f"Ошибка: {e}"],
            "processing_time": time.time() - start,
            "status": "error",
        }


@app.post("/debug")
def debug(request: Request):
    start = time.time()
    try:
        result = gift_agent(request.description, verbose=False)
        result["processing_time"] = time.time() - start
        return result
    except Exception as e:
        logger.exception("❌ Ошибка debug")
        return {"status": "error", "error": str(e), "processing_time": time.time() - start}


Writing api_qwen_kaggle.py


In [5]:
import os
import traceback
import torch

print("GPU available:", torch.cuda.is_available())
print("DATA_PATH:", os.environ.get("DATA_PATH"))
print("DATA_PATH exists:", os.path.exists(os.environ.get("DATA_PATH", "")))

try:
    import rag_pipeline_qwen_kaggle  # здесь сразу увидим реальную ошибку, если она есть
    print("✅ rag_pipeline_qwen_kaggle импортирован успешно")
except Exception as e:
    print("❌ Ошибка при импорте rag_pipeline_qwen_kaggle")
    traceback.print_exc()
    raise


GPU available: True
DATA_PATH: /kaggle/input/datasets/aleksanderviktorov/gift-genius-dataset/gifts_dataset_ru_plus.csv
DATA_PATH exists: True
🎁 GiftGenius | Kaggle backend without external LLM API
LLM model: Qwen/Qwen3.5-4B
LLM device: cuda
Thinking mode: False
Retrieval: pure-python BM25 (no sentence-transformers / no scipy)

📊 Загрузка датасета...
✅ Подарков в базе: 305

🔍 Построение BM25 индекса...
✅ BM25 готов | docs=305 | avgdl=32.4

🤖 Загрузка Qwen3.5-4B в 4-bit...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

✅ Qwen загружен

✅ Backend готов: BM25 retrieval на CPU, Qwen 4-bit на GPU
✅ rag_pipeline_qwen_kaggle импортирован успешно


In [6]:
import requests
import uvicorn
from threading import Thread
import time
import os
import traceback

def run_server():
    try:
        uvicorn.run("api_qwen_kaggle:app", host="0.0.0.0", port=8000, log_level="info")
    except Exception:
        print("❌ Uvicorn crashed:")
        traceback.print_exc()

os.system("pkill -f uvicorn || true")
os.system("pkill -f ngrok || true")
time.sleep(2)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

ready = False
for step in range(120):  # до ~10 минут ожидания
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        print("✅ FastAPI сервер поднят:", r.json())
        ready = True
        break
    except Exception:
        if step % 6 == 0:
            print(f"⏳ Ждем запуск сервера... {step*5} сек")
        time.sleep(5)

if not ready:
    raise RuntimeError(
        "Сервер не поднялся. Смотрите traceback выше. "
        "Если ошибка уже не про scipy/sklearn, пришлите новый traceback."
    )


⏳ Ждем запуск сервера... 0 сек


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:36954 - "GET /health HTTP/1.1" 200 OK
✅ FastAPI сервер поднят: {'status': 'healthy', 'timestamp': 1773763139.0155435}


In [7]:
from pyngrok import ngrok
import os

ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])
ngrok.kill()

public_url = ngrok.connect(8000)
print("=" * 60)
print("🎁 GIFT GENIUS + QWEN3.5-4B (4-bit) ГОТОВ")
print("=" * 60)
print("🌐 BASE URL:", public_url.public_url)
print("📌 HEALTH:", f"{public_url.public_url}/health")
print("📌 GENERATE:", f"{public_url.public_url}/generate")
print("=" * 60)

INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
INFO:pyngrok.ngrok:Opening tunnel named: http-8000-58be6c73-b806-4e15-b966-6b756319d753
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="FIPS 140 mode" enabled=false
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="tunnel sessi

🎁 GIFT GENIUS + QWEN3.5-4B (4-bit) ГОТОВ
🌐 BASE URL: https://lucio-lucent-jurnee.ngrok-free.dev
📌 HEALTH: https://lucio-lucent-jurnee.ngrok-free.dev/health
📌 GENERATE: https://lucio-lucent-jurnee.ngrok-free.dev/generate


INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg=end pg=/api/tunnels id=43b96ed9e1a9d6e3 status=201 dur=308.649229ms


In [8]:
from pyngrok import ngrok
import os

ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])
ngrok.kill()

public_url = ngrok.connect(8000)
print("=" * 60)
print("🎁 GIFT GENIUS + QWEN3.5-4B (4-bit) ГОТОВ")
print("=" * 60)
print("🌐 BASE URL:", public_url.public_url)
print("📌 HEALTH:", f"{public_url.public_url}/health")
print("📌 GENERATE:", f"{public_url.public_url}/generate")
print("=" * 60)

INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
INFO:pyngrok.process:Killing ngrok process: 203
INFO:pyngrok.ngrok:Opening tunnel named: http-8000-d71ac006-b9d1-4943-9083-435de1fa5d89
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="FIPS 140 mode" enabled=false
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:09+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:10+0000 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=202

🎁 GIFT GENIUS + QWEN3.5-4B (4-bit) ГОТОВ
🌐 BASE URL: https://lucio-lucent-jurnee.ngrok-free.dev
📌 HEALTH: https://lucio-lucent-jurnee.ngrok-free.dev/health
📌 GENERATE: https://lucio-lucent-jurnee.ngrok-free.dev/generate


INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:10+0000 lvl=info msg=end pg=/api/tunnels id=43b96ed9e1a9d6e3 status=201 dur=316.305165ms


In [9]:
import requests

# Сначала убеждаемся, что сервер реально жив
health = requests.get("http://127.0.0.1:8000/health", timeout=10)
print("health:", health.status_code, health.json())

test_query = "Маме 50 лет, любит готовить и читать. Бюджет до 3000 рублей. Не любит сладкое и украшения."

response = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"description": test_query},
    timeout=1000,
)
print("status:", response.status_code)
result = response.json()
print("keys:", result.keys())
for i, idea in enumerate(result.get("ideas", []), 1):
    print("\n" + "=" * 40)
    print(f"ИДЕЯ {i}")
    print("=" * 40)
    print(idea)


INFO:     127.0.0.1:44992 - "GET /health HTTP/1.1" 200 OK


INFO:api_qwen_kaggle:📨 Запрос: Маме 50 лет, любит готовить и читать. Бюджет до 3000 рублей. Не любит сладкое и украшения.


health: 200 {'status': 'healthy', 'timestamp': 1773763150.4883132}


INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:34+0000 lvl=info msg="join connections" obj=join id=e736f39bc71c l=127.0.0.1:8000 r=83.242.179.106:20890


INFO:     83.242.179.106:0 - "GET /health HTTP/1.1" 200 OK


INFO:api_qwen_kaggle:📨 Запрос: Подруга, 30 лет, йога, веганство, любит читать
INFO:pyngrok.process.ngrok:t=2026-03-17T15:59:38+0000 lvl=info msg="join connections" obj=join id=0b5e5f9c6902 l=127.0.0.1:8000 r=83.242.179.106:20906


INFO:     127.0.0.1:44994 - "POST /generate HTTP/1.1" 200 OK
status: 200
keys: dict_keys(['ideas', 'processing_time', 'status'])

ИДЕЯ 1
ПОДАРОК: Профессиональная кофемолка для зерен (например, модель Krups KM180 или аналогичный бренд в сегменте до 3000 руб.)
ПОЧЕМУ: Это идеальный инструмент для создания «шедевров» в кулинарии, позволяющий получать свежую молотую кофе, который значительно улучшает вкус напитка и добавляет момент рутинной удовольствия в процесс готовки.
МОМЕНТ: Вечерняя ритуализация начала дня или завершения готовки, когда мама сама заваривает ароматный напиток в своей любимой чашке, погружаясь в состояние спокойного уюта.
ЧЕМ НЕ БАНАЛЬНО: В отличие от простых блендеров или наборов ножей, кофемолка напрямую влияет на качество продукта и является профессиональным инструментом для гурманов, а не просто кухонным украшением.

ИДЕЯ 2
ПОДАРОК: Красная книга серии «Идеальная библиотека» (например, «Искусство быть счастливым» или сборник эссе)
ПОЧЕМУ: Такая книга соответствует 

## Как подключить локальный интерфейс

1. Скопируйте `BASE URL` из ячейки выше.
2. Локально запустите `app_qwen_local.py`.
3. Вставьте этот URL в боковую панель Streamlit.
4. Все: UI работает локально, а генерация идет на GPU в Kaggle.
